# 14. Full usable feature pool, then feature selection

This notebook starts with the complete usable representation of the prior candidate feature list for Dynamic Panel FE OLS. It then screens groups by dropping one group at a time. Notebook 13 is retained unchanged.

The pool contains core case history, 7/14-day means and variability, acceleration/jerk, local population-adjusted rates, sparse-case history, both pressure windows, target weekday and month. All comparisons use a common valid-population/pressure cohort (585 regions in the verified run). Cohort changes are audited, not caused by model performance.

Sums and means are exact duplicates; week-over-week change is a linear combination of the means; velocity is redundant with today/yesterday. Their information is represented in FULL. Static population/density and commuting summaries are absorbed by region FE: retain them for tree models, rather than pretending their FE coefficients are identifiable. Route weights are auxiliary inputs to pressure.

Screening is not final model selection. Use validation improvement, uncertainty and horizon stability together, jointly refit any proposed reduced set, and check window sensitivity. Do not delete features only because coefficient p>0.05. All inference is exploratory on repeatedly inspected validation data. No test data are scored here.


In [1]:
"""Validation-only feature ablations and exploratory inference for NJ onset counts.

Requires numpy/pandas. Optional matplotlib creates report figures. No GMM tests.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd

REPO=Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
SOURCE=REPO/'nj_daily_feature_outputs'/'nj_daily_features.csv.gz'

BASE=['cases_today','cases_lag_1','cases_lag_7']
HISTORY=BASE+['cases_mean_7d']
EXTENDED=HISTORY+['cases_lag_14','case_std_7d']
ACC=EXTENDED+['case_acceleration']
DYNAMICS=ACC+['case_jerk']
SPECIFICATIONS={
    'B0_basic':BASE,
    'H0_weekly_mean':HISTORY,
    'H1_history_extended':EXTENDED,
    'D1_acceleration':ACC,
    'D2_jerk':DYNAMICS,
    'P0_basic_pressure7':BASE+['nj_covid_pressure_7d'],
    'P1_rich_pressure7':DYNAMICS+['nj_covid_pressure_7d'],
    'P2_rich_pressure14':DYNAMICS+['nj_covid_pressure_14d'],
}
COMPARISONS=[
    ('B0_basic','H0_weekly_mean','weekly_mean'),
    ('H0_weekly_mean','H1_history_extended','lag14_and_volatility'),
    ('H1_history_extended','D1_acceleration','acceleration'),
    ('D1_acceleration','D2_jerk','jerk'),
    ('B0_basic','P0_basic_pressure7','pressure7_basic'),
    ('D2_jerk','P1_rich_pressure7','pressure7_rich'),
    ('D2_jerk','P2_rich_pressure14','pressure14_rich'),
    ('P1_rich_pressure7','P2_rich_pressure14','pressure14_vs_pressure7'),
]


def two_way_meat(scores,n_parameters):
    """CR1 two-way cluster sandwich meat: region + date - region/date cell."""
    N,T,p=scores.shape
    n=N*T
    if min(N,T)<=1 or n<=n_parameters:
        return np.full((p,p),np.nan)
    base=(n-1)/(n-n_parameters)
    by_region=scores.sum(axis=1)
    by_date=scores.sum(axis=0)
    cells=scores.reshape(-1,p)
    return base*(N/(N-1)*(by_region.T@by_region)+T/(T-1)*(by_date.T@by_date)-n/(n-1)*(cells.T@cells))


def fit_fe(x,y):
    N,T,p=x.shape
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm=x.mean(axis=1); ym=y.mean(axis=1)
    xc=x-xm[:,None,:]; yc=y-ym[:,None]
    scale=np.sqrt(np.mean(xc*xc,axis=(0,1)))
    scale=np.where(scale>1e-12,scale,1)
    z=xc/scale
    zf=z.reshape(-1,p)
    b,_,rank,sv=np.linalg.lstsq(zf,yc.reshape(-1),rcond=None)
    beta=b/scale; alpha=ym-xm@beta
    residual=yc-np.einsum('ntp,p->nt',z,b)
    if rank==p:
        bread=np.linalg.pinv(zf.T@zf,rcond=1e-12)
        covariance=bread@two_way_meat(z*residual[:,:,None],N+p)@bread
        variance=np.diag(covariance)/(scale*scale)
        se=np.full(p,np.nan)
        positive=np.isfinite(variance)&(variance>0)
        se[positive]=np.sqrt(variance[positive])
    else:
        se=np.full(p,np.nan)
    pvalue=np.array([math.erfc(abs(v/s)/math.sqrt(2)) if np.isfinite(s) and s>0 else np.nan for v,s in zip(beta,se)])
    condition=float(sv[0]/sv[-1]) if len(sv) and sv[-1]>0 else np.inf
    return beta,alpha,se,pvalue,int(rank),condition


def hac_mean_test(d,bandwidth=14):
    """Two-sided asymptotic test of zero mean paired DATE-level loss difference."""
    d=np.asarray(d,float)
    assert np.isfinite(d).all()
    T=len(d); L=min(int(bandwidth),T-1)
    mean=float(d.mean()); c=d-mean
    longvar=float(c@c/T)
    for lag in range(1,L+1):
        longvar+=2*(1-lag/(L+1))*float(c[lag:]@c[:-lag]/T)
    variance=longvar/T*T/(T-1)
    if variance<=0 or not np.isfinite(variance):
        return mean,np.nan,np.nan
    se=math.sqrt(variance)
    return mean,se,math.erfc(abs(mean/se)/math.sqrt(2))


def holm_adjust(p):
    p=np.asarray(p,float); adjusted=np.full(p.shape,np.nan)
    good=np.flatnonzero(np.isfinite(p)); ordered=good[np.argsort(p[good])]
    if len(ordered):
        values=np.maximum.accumulate((len(ordered)-np.arange(len(ordered)))*p[ordered])
        adjusted[ordered]=np.minimum(values,1)
    return adjusted


def block_indices(T,block_length,repetitions,seed):
    """Noncircular moving-block resampling; one date index shared by all regions/horizons."""
    assert 1<=block_length<=T
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block_length+1,size=(repetitions,math.ceil(T/block_length)))
    return (starts[:,:,None]+np.arange(block_length)[None,None,:]).reshape(repetitions,-1)[:,:T]


def residual_correlation(errors,lag):
    a=errors[:,lag:]; b=errors[:,:-lag]
    a=a-a.mean(axis=1,keepdims=True); b=b-b.mean(axis=1,keepdims=True)
    den=np.sqrt(np.sum(a*a,axis=1)*np.sum(b*b,axis=1))
    value=np.full(len(a),np.nan)
    good=den>0; value[good]=np.sum(a[good]*b[good],axis=1)/den[good]
    return value


def self_checks():
    # FWL estimates and cluster covariance versus an explicit dummy design.
    rng=np.random.default_rng(729)
    N,T,p=5,12,2
    x=rng.normal(size=(N,T,p)); y=x@np.array([.5,-.25])+np.arange(N)[:,None]+rng.normal(size=(N,T))
    beta,alpha,se,pv,rank,_=fit_fe(x,y)
    design=np.column_stack([np.repeat(np.eye(N),T,axis=0),x.reshape(-1,p)])
    full=np.linalg.lstsq(design,y.reshape(-1),rcond=None)[0]
    assert np.allclose(full[:N],alpha) and np.allclose(full[N:],beta)
    e=y.reshape(-1)-design@full
    inv=np.linalg.pinv(design.T@design)
    covariance=inv@two_way_meat((design*e[:,None]).reshape(N,T,N+p),N+p)@inv
    positive=np.diag(covariance)[N:]>0
    assert np.allclose(se[positive]**2,np.diag(covariance)[N:][positive])
    assert np.allclose(holm_adjust([.01,.04,.03]),[.03,.06,.06])
    independent=rng.normal(size=100)
    _,s,_=hac_mean_test(independent,0)
    assert np.isclose(s,np.std(independent,ddof=1)/10)



GROUPS = {
    'core_history': ['cases_today','cases_lag_1','cases_lag_7','cases_lag_14'],
    'rolling_level': ['cases_mean_7d','cases_mean_14d'],
    'volatility': ['case_std_7d','case_std_14d'],
    'dynamics': ['case_acceleration','case_jerk'],
    'local_rates': ['case_rate_today_per_100k','case_rate_7d_per_100k','case_rate_14d_per_100k'],
    'sparse_history': ['days_since_last_observed_case_censored','no_previous_observed_case'],
    'pressure': ['nj_covid_pressure_7d','nj_covid_pressure_14d'],
    'month': [f'target_month_{k}' for k in range(2,13)],
}
WEEKDAY=[f'target_weekday_{k}' for k in range(2,8)]

def independent_columns(x):
    # Training-only rank filtering. Removes algebraic duplicates and calendar
    # categories absent from the current window; not predictive feature selection.
    z=x-x.mean(axis=1,keepdims=True)
    norms=np.sqrt(np.mean(z*z,axis=(0,1)))
    selected=[]; q=[]
    for j in range(z.shape[2]):
        if norms[j]<1e-10: continue
        v=(z[:,:,j]/norms[j]).reshape(-1).copy()
        for _ in range(2):
            for u in q:v-=u*(u@v)
        length=np.linalg.norm(v)
        if length>1e-7*np.sqrt(v.size):
            selected.append(j);q.append(v/length)
    return selected

def run_full_selection(source,output,windows=(30,),validation_start='2021-01-01',validation_end='2021-12-31',bootstrap_repetitions=2000):
    self_checks()
    source=Path(source); output=Path(output)
    folder=output/('full_pool_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    folder.mkdir(parents=True)
    fields=sorted(set(sum(GROUPS.values(),[]))-set(GROUPS['month'])-set(GROUPS['sparse_history']))
    extra=['zip','date','population','days_since_last_observed_case','pressure_definition']
    p=pd.read_csv(source,usecols=fields+extra,dtype={'zip':'string'},parse_dates=['date'])
    assert not p.duplicated(['zip','date']).any()
    assert p.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    ids=np.array(sorted(p.zip.unique())); dates=pd.date_range(p.date.min(),p.date.max())
    def mat(c):return p.pivot(index='zip',columns='date',values=c).reindex(index=ids,columns=dates).to_numpy(float)
    population=mat('population')[:,0]
    eligible=(population>0)&np.isfinite(population)&np.isfinite(mat('nj_covid_pressure_14d')[:,13])
    pd.DataFrame({'zip':ids,'population':population,'eligible':eligible,
        'reason':np.where(eligible,'included',np.where(~(population>0),'invalid_population_for_local_rates','no_usable_external_pressure'))}).to_csv(folder/'cohort_audit.csv',index=False)
    matrices={c:mat(c)[eligible] for c in fields}
    gap=mat('days_since_last_observed_case')[eligible]
    matrices['no_previous_observed_case']=(~np.isfinite(gap)).astype(float)
    # Before any observed positive, duration is right-censored at panel history.
    matrices['days_since_last_observed_case_censored']=np.where(np.isfinite(gap),gap,np.arange(len(dates))[None,:]+1)
    ids=ids[eligible]; n=len(ids); counts=matrices['cases_today']
    assert n>0
    origins=np.flatnonzero((dates>=pd.Timestamp(validation_start))&((dates+pd.Timedelta(days=3))<=pd.Timestamp(validation_end)))
    assert len(origins)>28
    full=sum(GROUPS.values(),[])+WEEKDAY
    specifications={'FULL':full,'BASIC':['cases_today','cases_lag_1','cases_lag_7']+WEEKDAY}
    for group,features in GROUPS.items():
        if group!='core_history':specifications['DROP_'+group]=[f for f in full if f not in features]
    catalog=[]
    for group,features in GROUPS.items():
        for f in features:catalog.append({'feature':f,'group':group,'FE_status':'candidate'})
    for f in WEEKDAY:catalog.append({'feature':f,'group':'weekday','FE_status':'forced_calendar_control'})
    exclusions={
        'cases_sum_7d; cases_sum_14d':'Exactly 7/14 times rolling means; represented by means.',
        'week_over_week_change':'Exactly 14*mean_7d - 14*mean_14d; redundant with both means.',
        'case_velocity':'Exactly cases_today - cases_lag_1.',
        'origin weekday/month':'Use target calendar categories as the prespecified calendar representation.',
        'population; land_area_sq_km; population_density_per_sq_km':'Static regional columns absorbed by full region FE; retain for tree models.',
        'resident_jobs_nj_network; workplace_jobs_nj_network; inbound_jobs_external_nj; outbound_jobs_external_nj; internal_jobs; external_inbound_share_nj_network; external_outbound_share_nj_network':'Static regional columns absorbed by full region FE; retain for tree models.',
        'inbound_weight_nj; outbound_weight_nj':'Route-level fields, not one predictor per region-day. Inbound information enters pressure.',
        'coverage flags; mapping status; source metadata; person diagnostic counts':'Audit fields, not prespecified predictors or validated outcomes.',
    }
    pd.DataFrame(catalog).to_csv(folder/'candidate_features.csv',index=False)
    pd.DataFrame(exclusions.items(),columns=['feature_or_family','reason']).to_csv(folder/'feature_representation_decisions.csv',index=False)
    protocol={'source':str(source.resolve()),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'regions':n,'forecast_dates':len(origins),'windows':list(windows),'horizons':[1,2,3],
        'retrain_every_days':7,'specifications':specifications,'test_evaluated':False,
        'selection':'One-pass grouped backward screening: FULL vs each group removed. Validation evidence only; no final selected-model claim. Core history and weekday retained.',
        'limitations':'Repeatedly inspected retrospective onset validation; exploratory. No real-time report availability, causal inference or post-selection-valid p-values.'}
    (folder/'protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
    metrics=[]; logs=[]; coefs=[]; losses=[]
    for window in windows:
        preds=np.full((len(specifications),3,n,len(origins)),np.nan)
        actual=np.stack([counts[:,origins+h] for h in (1,2,3)])
        for h in (1,2,3):
            target=dates+pd.Timedelta(days=h)
            for k in range(2,8):matrices[f'target_weekday_{k}']=np.broadcast_to((target.dayofweek==k-1).astype(float),(n,len(dates)))
            for k in range(2,13):matrices[f'target_month_{k}']=np.broadcast_to((target.month==k).astype(float),(n,len(dates)))
            for m,(name,features) in enumerate(specifications.items()):
                x=np.stack([matrices[c] for c in features],axis=2)
                for first in range(0,len(origins),7):
                    block=origins[first:first+7];fit=int(block[0]);last=fit-h
                    train=np.arange(last-window+1,last+1)
                    assert train[0]>=14 and train[-1]+h<=fit
                    xt=x[:,train]
                    assert np.isfinite(xt).all() and np.isfinite(x[:,block]).all()
                    keep=independent_columns(xt)
                    b,a,se,pv,rank,condition=fit_fe(xt[:,:,keep],counts[:,train+h])
                    assert rank==len(keep)
                    yhat=np.einsum('ntp,p->nt',x[:,block][:,:,keep],b)+a[:,None]
                    preds[m,h-1,:,first:first+len(block)]=yhat
                    logs.append({'window':window,'horizon':h,'specification':name,'fit_date':dates[fit],
                        'latest_training_target':dates[train[-1]+h],'training_origin_start':dates[train[0]],
                        'training_origin_end':dates[train[-1]],'regions':n,'rank':rank,'condition':condition,
                        'removed_by_training_rank':';'.join(f for j,f in enumerate(features) if j not in keep)})
                    for j,beta,s,prob in zip(keep,b,se,pv):coefs.append({'window':window,'horizon':h,'specification':name,'fit_date':dates[fit],'feature':features[j],'estimate':beta,'se_cluster_approx':s,'p_approx_exploratory':prob})
                print(f'window={window}, horizon={h}, {name} completed',flush=True)
        assert np.isfinite(preds).all()
        error=np.maximum(preds,0)-actual[None]
        for m,name in enumerate(specifications):
            for hh in range(3):
                e=error[m,hh]
                metrics.append({'window':window,'specification':name,'horizon':hh+1,'regions':n,'dates':len(origins),'MAE':np.abs(e).mean(),'RMSE':np.sqrt((e*e).mean()),'bias':e.mean()})
                for d,ae,mse in zip(dates[origins],np.abs(e).mean(axis=0),(e*e).mean(axis=0)):losses.append({'window':window,'specification':name,'horizon':hh+1,'date':d,'MAE':ae,'MSE':mse})
        np.savez_compressed(folder/f'predictions_window_{window}d.npz',raw_predictions=preds,actual_cases=actual,models=np.array(list(specifications)),zip=ids,date=dates[origins].to_numpy(),horizons=np.array([1,2,3]))
    metrics=pd.DataFrame(metrics);daily=pd.DataFrame(losses);logs=pd.DataFrame(logs)
    assert (pd.to_datetime(logs.latest_training_target)<=pd.to_datetime(logs.fit_date)).all()
    ranking=metrics.groupby(['window','specification'],as_index=False)[['MAE','RMSE','bias']].mean().sort_values(['window','MAE'])
    comparisons=[]
    for window in windows:
        table=daily.loc[daily.window.eq(window)].groupby(['specification','date']).MAE.mean().unstack(0)
        draws=block_indices(len(table),14,bootstrap_repetitions,20261027+window)
        for name in specifications:
            if name=='FULL':continue
            d=(table.FULL-table[name]).to_numpy()
            delta,se,pv=hac_mean_test(d,14);lo,hi=np.quantile(d[draws].mean(axis=1),[.025,.975])
            comparisons.append({'window':window,'candidate':name,'delta_MAE_FULL_minus_candidate':delta,'ci95_low':lo,'ci95_high':hi,'p_HAC_raw':pv,
                'interpretation':'Positive means reduced candidate has lower MAE than FULL.'})
    comparisons=pd.DataFrame(comparisons);comparisons['p_Holm']=holm_adjust(comparisons.p_HAC_raw)
    metrics.to_csv(folder/'metrics_by_horizon.csv',index=False);ranking.to_csv(folder/'validation_ranking.csv',index=False)
    comparisons.to_csv(folder/'group_selection_evidence.csv',index=False);daily.to_csv(folder/'daily_losses.csv',index=False)
    logs.to_csv(folder/'training_log.csv',index=False);pd.DataFrame(coefs).to_csv(folder/'coefficients_exploratory.csv',index=False)
    report='# Full candidate pool and grouped feature screening\n\n'
    report+=f'Common cohort: {n} regions. Validation only, horizons 1/2/3, weekly refits. No test evaluated. FULL is the complete usable FE representation; metadata, exact duplicates and static FE-absorbed columns are audited separately.\n\n'
    report+='Start with validation_ranking.csv and group_selection_evidence.csv. Positive FULL-minus-candidate delta favors the smaller model. Group removal is assessed one group at a time: do not assume multiple individually removable groups can all be deleted together without refitting. No final feature set is automatically selected.\n\n'
    report+='P-values use paired DATE-level loss differences with HAC bandwidth 14 and Holm adjustment across this run. CIs use 2,000 14-day date-block resamples of fixed saved forecasts. They are exploratory and pointwise. Coefficient p-values are approximate clustered FE OLS calculations, not causal, and dynamic FE bias remains uncorrected.\n\n'
    report+='Calendar rank filtering uses training data only. A month absent from a training window has no identified effect; its coefficient is omitted for that refit. The month effect may therefore be unavailable at the start of a new month. No-prior-case gaps are censored at available panel history and explicitly flagged. Undefined local rates are handled through a fixed valid-population cohort, never imputed as observed zeros.\n\n'
    report+='Next: choose a parsimonious candidate using MAE/RMSE, uncertainty, and horizon stability; jointly refit the chosen combination; check 60-day sensitivity. Do not select only by coefficient p-values. Earlier test results have already been inspected.\n'
    (folder/'README.md').write_text(report,encoding='utf-8')
    print(ranking.to_string(index=False));print('Saved to:',folder.resolve())
    return ranking,comparisons,folder


## Run full model and grouped screening

Run from Weikun. Default: 30-day rolling window, weekly refit, 2021 validation, direct horizons 1/2/3. Change windows to (30,60) for the sensitivity run. Every run saves into its own output folder. Dependencies: numpy and pandas.


In [2]:
SOURCE = Path.cwd() / 'nj_daily_feature_outputs' / 'nj_daily_features.csv.gz'
OUTPUT = Path.cwd() / 'full_feature_selection_outputs'
ranking, evidence, RUN_FOLDER = run_full_selection(SOURCE, OUTPUT, windows=(30,))
display(ranking)
display(evidence)
print(RUN_FOLDER.resolve())


window=30, horizon=1, FULL completed
window=30, horizon=1, BASIC completed
window=30, horizon=1, DROP_rolling_level completed
window=30, horizon=1, DROP_volatility completed
window=30, horizon=1, DROP_dynamics completed
window=30, horizon=1, DROP_local_rates completed
window=30, horizon=1, DROP_sparse_history completed
window=30, horizon=1, DROP_pressure completed
window=30, horizon=1, DROP_month completed
window=30, horizon=2, FULL completed
window=30, horizon=2, BASIC completed
window=30, horizon=2, DROP_rolling_level completed
window=30, horizon=2, DROP_volatility completed
window=30, horizon=2, DROP_dynamics completed
window=30, horizon=2, DROP_local_rates completed
window=30, horizon=2, DROP_sparse_history completed
window=30, horizon=2, DROP_pressure completed
window=30, horizon=2, DROP_month completed
window=30, horizon=3, FULL completed
window=30, horizon=3, BASIC completed
window=30, horizon=3, DROP_rolling_level completed
window=30, horizon=3, DROP_volatility completed
window

,window,specification,MAE,RMSE,bias
4,30,DROP_pressure,1.110063,1.921807,0.020778
3,30,DROP_month,1.113731,1.913848,0.046200
6,30,DROP_sparse_history,1.116508,1.915866,0.067414
7,30,DROP_volatility,1.116888,1.915518,0.069248
8,30,FULL,1.117473,1.915829,0.068172
1,30,DROP_dynamics,1.117610,1.916299,0.068298
2,30,DROP_local_rates,1.118870,1.917928,0.069440
5,30,DROP_rolling_level,1.127262,1.926579,0.067715
0,30,BASIC,1.142268,1.995349,-0.043577


,window,candidate,delta_MAE_FULL_minus_candidate,ci95_low,ci95_high,p_HAC_raw,interpretation,p_Holm
0,30,BASIC,-0.024795,-0.075739,0.002589,0.248380,Positive means reduced candidate has lower MAE...,1.000000
1,30,DROP_rolling_level,-0.009790,-0.014281,-0.003146,0.005009,Positive means reduced candidate has lower MAE...,0.040069
2,30,DROP_volatility,0.000585,-0.001138,0.002383,0.501788,Positive means reduced candidate has lower MAE...,1.000000
3,30,DROP_dynamics,-0.000137,-0.000392,0.000264,0.494647,Positive means reduced candidate has lower MAE...,1.000000
4,30,DROP_local_rates,-0.001397,-0.002529,-0.000551,0.007653,Positive means reduced candidate has lower MAE...,0.053570
5,30,DROP_sparse_history,0.000965,-0.000056,0.002273,0.104872,Positive means reduced candidate has lower MAE...,0.629231
6,30,DROP_pressure,0.007410,-0.019709,0.022637,0.573068,Positive means reduced candidate has lower MAE...,1.000000
7,30,DROP_month,0.003742,-0.008879,0.004469,0.548063,Positive means reduced candidate has lower MAE...,1.000000


C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\full_feature_selection_outputs\full_pool_20261026T145123986715Z


## Files to inspect

* candidate_features.csv and feature_representation_decisions.csv: full pool and treatment of every prior feature family.
* cohort_audit.csv: inclusion/exclusion independent of prediction performance.
* validation_ranking.csv and metrics_by_horizon.csv: full vs reduced models.
* group_selection_evidence.csv: paired improvement, pointwise block CI, HAC p and Holm p.
* training_log.csv: timing, rank checks and unavailable training calendar effects.
* coefficients_exploratory.csv: approximate coefficient inference; no dynamic FE bias correction.
* predictions_window_30d.npz: saved raw predictions and outcomes, same axes as Notebook 13.

Residual Moran's I remains a separate spatial diagnostic pending a real spatial weights definition. It is not replaced by feature selection.
